# 標準住宅・気密6条件 × 断熱等級4〜7

外皮隙間倍率×0、×0.5、×1、×2、×4、×8（公称C値0、0.30、0.59、1.19、2.37、4.74 cm²/m²）と基礎断熱仕様 foundation4〜foundation7 の全24条件を比較します。室内建具の隙間、換気設備、気象、内部発熱、全館空調1台・7系統、設定温度と運転スケジュールは共通です。

等級によって外壁・天井・開口部・基礎の仕様と窓の日射取得特性を変えます。基礎外壁XPSは等級4〜7で47.3、102.4、122.4、157.4 mmです。年間は12月15日から助走し、1月1日からの8,760時間を集計します。負荷は空調機の処理熱量です。

収録済みの計算結果　年間 8/24条件、冬の168時間 24/24条件、夏の168時間 24/24条件

In [ ]:
%pip -q install "git+https://github.com/iguchi-lab/vtsimnx.git@4e957e94f0c13f000b986178b49dee4d200942c8" japanize-matplotlib

In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path

from google.colab import drive, userdata
from IPython.display import display
import japanize_matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import vtsimnx as vt

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/sim検討")
WEATHER_FILE = DATA_DIR / "3639999.has"
WIND_COEFFICIENT_FILE = DATA_DIR / "風圧係数.csv"


import hashlib
import json

RUN_ANNUAL = True  # Falseで保存済み結果の表示だけを行う
GRADES_TO_RUN = (4, 5, 6, 7)
MULTIPLIERS_TO_RUN = (0, 0.5, 1, 2, 4, 8)
RESULT_DIR = DATA_DIR / "全館空調_気密断熱24条件"

In [ ]:
h = vt.standard_house
FACES = ("東", "南", "西", "北")
df_i = vt.read_hasp(WEATHER_FILE)
df_coef = pd.read_csv(WIND_COEFFICIENT_FILE, index_col=0)
df_coef.index = pd.to_numeric(df_coef.index, errors="raise")
if df_coef.index.has_duplicates:
    raise ValueError("風圧係数表に重複した風向があります。")
wind_table = df_coef.loc[list(range(1, 17)), [face + "面" for face in FACES]]
if not np.isfinite(wind_table.to_numpy(dtype=float)).all():
    raise ValueError("風圧係数表の1～16方位に欠損または非有限値があります。")
pressures = h.facade_pressures(df_i, wind_table.astype(float))

solar = h.compute_solar(df_i["直達日射量"], df_i["水平面拡散日射量"])
nocturnal = h.compute_nocturnal(df_i["夜間放射量"])
ground_temperature = vt.ground_temperature_by_depth(
    depth_m=1.0,
    t_out=df_i["外気温"],
    solar_horizontal=solar["日射熱取得量（水平上向き）"],
    nocturnal_horizontal=df_i["夜間放射量"],
    deep_layer_depth_m=10.0,
    deep_layer_temp_c=10.0,
    thermal_conductivity_w_mk=1.5,
    volumetric_heat_capacity_j_m3k=2.2e6,
    solar_to_surface_temp_coeff=0.003,
    nocturnal_to_surface_temp_coeff=0.003,
    spinup=True,
)

layers = h.layers["default"]
# 元モデルの床層を使い、ホール床下のXPS層を基礎外壁側へ移す。
foundation_xps = layers["ホール-床下"].pop()
foundation_xps["t"] = 0.022
layers["基礎外壁"].append(foundation_xps)
surface_types = h.bind_surface_types(h.surface_types["default"], layers, solar=solar, nocturnal=nocturnal)

base_input = {
    "simulation": {
        "index": df_i.index,
        "warmup": {"start": h.WARMUP_START},
        "tolerance": {"ventilation": 1e-5, "thermal": 1e-3, "convergence": 1e-3},
    },
    "nodes": [
        *h.gap_nodes(df_i["外気温"], df_i["外気絶対湿度"] / 1000, pressures, central_air=True),
        {"key": "地下1m", "t": ground_temperature},
    ],
    "surfaces": h.compose_surfaces(h.surface_rows["default"], surface_types),
    "heat_source": h.heat_source["default"],
    "humidity_source": h.humidity_source["default"],
    "aircon": h.central_aircon(),
}

def make_input(multiplier):
    return {
        **base_input,
        "ventilation_branches": h.gap_network(multiplier, central_air=True),
    }

In [ ]:
GRADES = (4, 5, 6, 7)
HEATING = "暖房顕熱 [kWh]"
COOLING_SENSIBLE = "冷房顕熱 [kWh]"
COOLING_LATENT = "冷房潜熱 [kWh]"
COOLING = "冷房合計 [kWh]"
TOTAL = "暖冷房合計 [kWh]"
POWER = "空調消費電力量 [kWh]"
HEATING_UNMET = "暖房未達 >0.5 K [h]"
COOLING_UNMET = "冷房未達 >0.5 K [h]"
METRICS = [HEATING, COOLING_SENSIBLE, COOLING_LATENT, COOLING, TOTAL, POWER, HEATING_UNMET, COOLING_UNMET]

def make_grade_input(grade, gap_multiplier):
    model = f"foundation{grade}"
    materials = h.layers[model]
    types = h.bind_surface_types(h.surface_types[model], materials, solar=solar, nocturnal=nocturnal)
    config = make_input(gap_multiplier)
    config["surfaces"] = h.compose_surfaces(h.surface_rows[model], types)
    return config

def grade_loads(result, config):
    fields = {
        "sensible": "aircon_sensible_heat",
        "latent": "aircon_latent_heat",
        "power": "aircon_power",
    }
    index = config["simulation"]["index"]
    hours = (index[1] - index[0]).total_seconds() / 3600
    unit = config["aircon"][0]["key"]
    mode = pd.Series(config["aircon"][0]["mode"], index=index)
    series = {}
    for key, name in fields.items():
        values = result.get_series_df(name, raise_on_error=True)[unit]
        if not values.index.equals(index) or not np.isfinite(values.to_numpy()).all():
            raise ValueError(f"{name}: 評価期間の時刻が合わないか、欠損値があります。")
        series[key] = values.astype(float)
    heating = mode.eq(1)
    cooling = mode.eq(2)
    hourly = pd.DataFrame({
        HEATING: series["sensible"].abs().where(heating, 0) * hours / 1000,
        COOLING_SENSIBLE: series["sensible"].abs().where(cooling, 0) * hours / 1000,
        COOLING_LATENT: series["latent"].abs().where(cooling, 0) * hours / 1000,
        POWER: series["power"] * hours / 1000,
    })
    hourly[COOLING] = hourly[COOLING_SENSIBLE] + hourly[COOLING_LATENT]
    hourly[TOTAL] = hourly[HEATING] + hourly[COOLING]
    annual = hourly.sum().to_dict()
    unmet = result.get_series_df("aircon_temperature_unmet", raise_on_error=False)
    if (
        isinstance(unmet, pd.DataFrame) and unit in unmet and unmet.index.equals(index)
        and np.isfinite(unmet[unit].to_numpy()).all()
    ):
        annual[HEATING_UNMET] = (heating & unmet[unit].gt(0.5)).sum() * hours
        annual[COOLING_UNMET] = (cooling & unmet[unit].gt(0.5)).sum() * hours
    return annual, hourly.groupby(hourly.index.month).sum().rename_axis("月")

In [ ]:
REFERENCE_WEATHER_SHA = 'de98ac5d090e2f42ef004e8effbe70cc13ba9f29985a356bd4927252abfa4af6'
REFERENCE_WIND_SHA = '96e29e25061b390874c9ddb52ab869118b3b5aa0f79c32539bba8ee764451ad3'
VERIFIED_ANNUAL = json.loads("[\n  {\n    \"等級\": 4,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 5839.997444159984,\n    \"冷房顕熱 [kWh]\": 3127.8640447394846,\n    \"冷房潜熱 [kWh]\": 1142.99514679496,\n    \"冷房合計 [kWh]\": 4270.859191534444,\n    \"暖冷房合計 [kWh]\": 10110.856635694428,\n    \"空調消費電力量 [kWh]\": 3222.18972265625,\n    \"暖房未達 >0.5 K [h]\": 114,\n    \"冷房未達 >0.5 K [h]\": 52\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 1.0,\n    \"状態\": \"完了\",\n    \"基礎XPS [mm]\": 47.288888888888884,\n    \"暖房顕熱 [kWh]\": 5980.995122657776,\n    \"冷房顕熱 [kWh]\": 3148.9778396795987,\n    \"冷房潜熱 [kWh]\": 1143.6128049759866,\n    \"冷房合計 [kWh]\": 4292.5906446555855,\n    \"暖冷房合計 [kWh]\": 10273.585767313361,\n    \"空調消費電力量 [kWh]\": 3257.0980208435058,\n    \"暖房未達 >0.5 K [h]\": 111.0,\n    \"冷房未達 >0.5 K [h]\": 52.0\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 6138.319488176822,\n    \"冷房顕熱 [kWh]\": 3159.2778713886737,\n    \"冷房潜熱 [kWh]\": 1153.3745801633597,\n    \"冷房合計 [kWh]\": 4312.652451552033,\n    \"暖冷房合計 [kWh]\": 10450.971939728855,\n    \"空調消費電力量 [kWh]\": 3292.0200521850584,\n    \"暖房未達 >0.5 K [h]\": 108,\n    \"冷房未達 >0.5 K [h]\": 52\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 6548.817735517502,\n    \"冷房顕熱 [kWh]\": 3166.665470431566,\n    \"冷房潜熱 [kWh]\": 1183.01279604578,\n    \"冷房合計 [kWh]\": 4349.678266477346,\n    \"暖冷房合計 [kWh]\": 10898.496001994848,\n    \"空調消費電力量 [kWh]\": 3382.8490105895994,\n    \"暖房未達 >0.5 K [h]\": 98,\n    \"冷房未達 >0.5 K [h]\": 62\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 7483.253807011604,\n    \"冷房顕熱 [kWh]\": 3178.605231601119,\n    \"冷房潜熱 [kWh]\": 1279.8135733554363,\n    \"冷房合計 [kWh]\": 4458.4188049565555,\n    \"暖冷房合計 [kWh]\": 11941.672611968159,\n    \"空調消費電力量 [kWh]\": 3584.190300933838,\n    \"暖房未達 >0.5 K [h]\": 87,\n    \"冷房未達 >0.5 K [h]\": 51\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 1.0,\n    \"状態\": \"完了\",\n    \"基礎XPS [mm]\": 102.36842105263158,\n    \"暖房顕熱 [kWh]\": 3881.943920651436,\n    \"冷房顕熱 [kWh]\": 2980.2863013881447,\n    \"冷房潜熱 [kWh]\": 1153.4509500906468,\n    \"冷房合計 [kWh]\": 4133.737251478791,\n    \"暖冷房合計 [kWh]\": 8015.681172130227,\n    \"空調消費電力量 [kWh]\": 2787.3231921386714,\n    \"暖房未達 >0.5 K [h]\": 128.0,\n    \"冷房未達 >0.5 K [h]\": 58.0\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 1.0,\n    \"状態\": \"完了\",\n    \"基礎XPS [mm]\": 122.36842105263159,\n    \"暖房顕熱 [kWh]\": 2837.786586939573,\n    \"冷房顕熱 [kWh]\": 2556.998242822647,\n    \"冷房潜熱 [kWh]\": 1128.245201151371,\n    \"冷房合計 [kWh]\": 3685.243443974018,\n    \"暖冷房合計 [kWh]\": 6523.030030913591,\n    \"空調消費電力量 [kWh]\": 2501.519729385376,\n    \"暖房未達 >0.5 K [h]\": 134.0,\n    \"冷房未達 >0.5 K [h]\": 65.0\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 1.0,\n    \"状態\": \"完了\",\n    \"基礎XPS [mm]\": 157.36842105263156,\n    \"暖房顕熱 [kWh]\": 2432.914207913399,\n    \"冷房顕熱 [kWh]\": 1877.8477075102328,\n    \"冷房潜熱 [kWh]\": 1028.693024070263,\n    \"冷房合計 [kWh]\": 2906.5407315804955,\n    \"暖冷房合計 [kWh]\": 5339.454939493894,\n    \"空調消費電力量 [kWh]\": 2339.8654026184086,\n    \"暖房未達 >0.5 K [h]\": 161.0,\n    \"冷房未達 >0.5 K [h]\": 79.0\n  }\n]")
VERIFIED_WEEKS = json.loads("[\n  {\n    \"等級\": 4,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 355.0731343383789,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 355.0731343383789,\n    \"空調消費電力量 [kWh]\": 118.60151580810548,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 280.6463608932495,\n    \"冷房潜熱 [kWh]\": 125.32216938781738,\n    \"冷房合計 [kWh]\": 405.9685302810669,\n    \"暖冷房合計 [kWh]\": 405.9685302810669,\n    \"空調消費電力量 [kWh]\": 108.45648231506348,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 358.9748610534668,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 358.9748610534668,\n    \"空調消費電力量 [kWh]\": 119.34667514038085,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 281.98448437881467,\n    \"冷房潜熱 [kWh]\": 125.170578125,\n    \"冷房合計 [kWh]\": 407.1550625038147,\n    \"暖冷房合計 [kWh]\": 407.1550625038147,\n    \"空調消費電力量 [kWh]\": 109.03485008239745,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 361.6540314025879,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 361.6540314025879,\n    \"空調消費電力量 [kWh]\": 119.88699520874023,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 282.8631599121094,\n    \"冷房潜熱 [kWh]\": 125.5150840034485,\n    \"冷房合計 [kWh]\": 408.37824391555785,\n    \"暖冷房合計 [kWh]\": 408.37824391555785,\n    \"空調消費電力量 [kWh]\": 109.35537615966797,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 368.73554217529295,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 368.73554217529295,\n    \"空調消費電力量 [kWh]\": 121.0780251159668,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 284.2188073692322,\n    \"冷房潜熱 [kWh]\": 127.56286154174805,\n    \"冷房合計 [kWh]\": 411.7816689109802,\n    \"暖冷房合計 [kWh]\": 411.7816689109802,\n    \"空調消費電力量 [kWh]\": 109.84073419189453,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 387.3102360229492,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 387.3102360229492,\n    \"空調消費電力量 [kWh]\": 124.81926495361328,\n    \"暖房未達 >0.5 K [h]\": 1,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 285.95447731399537,\n    \"冷房潜熱 [kWh]\": 133.19519200515748,\n    \"冷房合計 [kWh]\": 419.14966931915285,\n    \"暖冷房合計 [kWh]\": 419.14966931915285,\n    \"空調消費電力量 [kWh]\": 110.50925314331054,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 426.50442248535154,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 426.50442248535154,\n    \"空調消費電力量 [kWh]\": 130.84972436523438,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 4,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 289.541926607132,\n    \"冷房潜熱 [kWh]\": 148.5307968978882,\n    \"冷房合計 [kWh]\": 438.0727235050202,\n    \"暖冷房合計 [kWh]\": 438.0727235050202,\n    \"空調消費電力量 [kWh]\": 112.7909144897461,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 240.26481578063965,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 240.26481578063965,\n    \"空調消費電力量 [kWh]\": 99.5270199584961,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 251.26158589363098,\n    \"冷房潜熱 [kWh]\": 123.00686498260498,\n    \"冷房合計 [kWh]\": 374.268450876236,\n    \"暖冷房合計 [kWh]\": 374.268450876236,\n    \"空調消費電力量 [kWh]\": 101.0985507659912,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 243.83145265197754,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 243.83145265197754,\n    \"空調消費電力量 [kWh]\": 100.2359234008789,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 252.47623658943175,\n    \"冷房潜熱 [kWh]\": 122.88467342376708,\n    \"冷房合計 [kWh]\": 375.3609100131988,\n    \"暖冷房合計 [kWh]\": 375.3609100131988,\n    \"空調消費電力量 [kWh]\": 101.6875366973877,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 246.3676849822998,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 246.3676849822998,\n    \"空調消費電力量 [kWh]\": 100.77614959716797,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 253.2986224656105,\n    \"冷房潜熱 [kWh]\": 123.1811713027954,\n    \"冷房合計 [kWh]\": 376.4797937684059,\n    \"暖冷房合計 [kWh]\": 376.4797937684059,\n    \"空調消費電力量 [kWh]\": 102.01261994934082,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 253.28305126953126,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 253.28305126953126,\n    \"空調消費電力量 [kWh]\": 101.92205920410156,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 254.65260492706298,\n    \"冷房潜熱 [kWh]\": 125.16313872909546,\n    \"冷房合計 [kWh]\": 379.8157436561585,\n    \"暖冷房合計 [kWh]\": 379.8157436561585,\n    \"空調消費電力量 [kWh]\": 102.1636795349121,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 272.55097007751465,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 272.55097007751465,\n    \"空調消費電力量 [kWh]\": 106.67588348388672,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 256.4475609169006,\n    \"冷房潜熱 [kWh]\": 130.49675078201295,\n    \"冷房合計 [kWh]\": 386.94431169891357,\n    \"暖冷房合計 [kWh]\": 386.94431169891357,\n    \"空調消費電力量 [kWh]\": 102.64131164550781,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 313.7284362792969,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 313.7284362792969,\n    \"空調消費電力量 [kWh]\": 114.09900555419922,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 5,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 260.1943149795532,\n    \"冷房潜熱 [kWh]\": 145.00242260837555,\n    \"冷房合計 [kWh]\": 405.19673758792874,\n    \"暖冷房合計 [kWh]\": 405.19673758792874,\n    \"空調消費電力量 [kWh]\": 104.52375282287598,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 178.72857615947723,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 178.72857615947723,\n    \"空調消費電力量 [kWh]\": 91.90921032714844,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 210.10620455551148,\n    \"冷房潜熱 [kWh]\": 118.7555530166626,\n    \"冷房合計 [kWh]\": 328.8617575721741,\n    \"暖冷房合計 [kWh]\": 328.8617575721741,\n    \"空調消費電力量 [kWh]\": 92.14498307800292,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 181.96133243656158,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 181.96133243656158,\n    \"空調消費電力量 [kWh]\": 92.69753396606446,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 211.2531527748108,\n    \"冷房潜熱 [kWh]\": 118.6968632965088,\n    \"冷房合計 [kWh]\": 329.95001607131957,\n    \"暖冷房合計 [kWh]\": 329.95001607131957,\n    \"空調消費電力量 [kWh]\": 92.73855448913574,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 184.09589393234253,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 184.09589393234253,\n    \"空調消費電力量 [kWh]\": 91.54202648925781,\n    \"暖房未達 >0.5 K [h]\": 3,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 212.03405688667297,\n    \"冷房潜熱 [kWh]\": 118.95675014495849,\n    \"冷房合計 [kWh]\": 330.99080703163145,\n    \"暖冷房合計 [kWh]\": 330.99080703163145,\n    \"空調消費電力量 [kWh]\": 93.0536183013916,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 190.83211632537842,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 190.83211632537842,\n    \"空調消費電力量 [kWh]\": 92.94973272705079,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 213.3963572292328,\n    \"冷房潜熱 [kWh]\": 120.81387790679932,\n    \"冷房合計 [kWh]\": 334.2102351360321,\n    \"暖冷房合計 [kWh]\": 334.2102351360321,\n    \"空調消費電力量 [kWh]\": 93.43769454956055,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 211.06641435241698,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 211.06641435241698,\n    \"空調消費電力量 [kWh]\": 99.27086434936524,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 215.25794666671752,\n    \"冷房潜熱 [kWh]\": 125.79144303894043,\n    \"冷房合計 [kWh]\": 341.049389705658,\n    \"暖冷房合計 [kWh]\": 341.049389705658,\n    \"空調消費電力量 [kWh]\": 93.8800754852295,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 253.69040043640138,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 253.69040043640138,\n    \"空調消費電力量 [kWh]\": 108.45923626708985,\n    \"暖房未達 >0.5 K [h]\": 2,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 6,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 219.06691762542724,\n    \"冷房潜熱 [kWh]\": 138.97730319213866,\n    \"冷房合計 [kWh]\": 358.04422081756593,\n    \"暖冷房合計 [kWh]\": 358.04422081756593,\n    \"空調消費電力量 [kWh]\": 95.0571611175537,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 2,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 148.3586916217804,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 148.3586916217804,\n    \"空調消費電力量 [kWh]\": 88.88251498413086,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 0,\n    \"公称C値 [cm²/m²]\": 0.0,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 153.97955775928497,\n    \"冷房潜熱 [kWh]\": 108.14154166793823,\n    \"冷房合計 [kWh]\": 262.1210994272232,\n    \"暖冷房合計 [kWh]\": 262.1210994272232,\n    \"空調消費電力量 [kWh]\": 81.7229934387207,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 3,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 151.22291736602784,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 151.22291736602784,\n    \"空調消費電力量 [kWh]\": 89.68818200683594,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 0.5,\n    \"公称C値 [cm²/m²]\": 0.3,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 154.8292709608078,\n    \"冷房潜熱 [kWh]\": 108.14051300811768,\n    \"冷房合計 [kWh]\": 262.96978396892547,\n    \"暖冷房合計 [kWh]\": 262.96978396892547,\n    \"空調消費電力量 [kWh]\": 80.88100117492675,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 4,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 153.62084549045562,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 153.62084549045562,\n    \"空調消費電力量 [kWh]\": 90.4023907775879,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 1,\n    \"公称C値 [cm²/m²]\": 0.59,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 155.5865917634964,\n    \"冷房潜熱 [kWh]\": 108.4230502204895,\n    \"冷房合計 [kWh]\": 264.0096419839859,\n    \"暖冷房合計 [kWh]\": 264.0096419839859,\n    \"空調消費電力量 [kWh]\": 81.27833798217773,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 4,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 159.68967478752137,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 159.68967478752137,\n    \"空調消費電力量 [kWh]\": 92.0111502380371,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 2,\n    \"公称C値 [cm²/m²]\": 1.19,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 156.95513284301757,\n    \"冷房潜熱 [kWh]\": 110.17473972320556,\n    \"冷房合計 [kWh]\": 267.1298725662231,\n    \"暖冷房合計 [kWh]\": 267.1298725662231,\n    \"空調消費電力量 [kWh]\": 81.76722145080566,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 4,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 180.22099836730956,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 180.22099836730956,\n    \"空調消費電力量 [kWh]\": 97.97235354614257,\n    \"暖房未達 >0.5 K [h]\": 4,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 4,\n    \"公称C値 [cm²/m²]\": 2.37,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 159.0093312149048,\n    \"冷房潜熱 [kWh]\": 114.50163087463379,\n    \"冷房合計 [kWh]\": 273.51096208953857,\n    \"暖冷房合計 [kWh]\": 273.51096208953857,\n    \"空調消費電力量 [kWh]\": 83.4004940032959,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 4,\n    \"期間\": \"7月31日〜8月6日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 224.99291592407226,\n    \"冷房顕熱 [kWh]\": 0.0,\n    \"冷房潜熱 [kWh]\": 0.0,\n    \"冷房合計 [kWh]\": 0.0,\n    \"暖冷房合計 [kWh]\": 224.99291592407226,\n    \"空調消費電力量 [kWh]\": 108.46299291992187,\n    \"暖房未達 >0.5 K [h]\": 6,\n    \"冷房未達 >0.5 K [h]\": 0,\n    \"期間\": \"1月1〜7日\"\n  },\n  {\n    \"等級\": 7,\n    \"倍率\": 8,\n    \"公称C値 [cm²/m²]\": 4.74,\n    \"状態\": \"完了\",\n    \"暖房顕熱 [kWh]\": 0.0,\n    \"冷房顕熱 [kWh]\": 162.7465279045105,\n    \"冷房潜熱 [kWh]\": 125.46870440673828,\n    \"冷房合計 [kWh]\": 288.21523231124877,\n    \"暖冷房合計 [kWh]\": 288.21523231124877,\n    \"空調消費電力量 [kWh]\": 83.8451524658203,\n    \"暖房未達 >0.5 K [h]\": 0,\n    \"冷房未達 >0.5 K [h]\": 4,\n    \"期間\": \"7月31日〜8月6日\"\n  }\n]")

## 年間計算と再開

元の気象ファイル・風圧係数表と、Colabのシークレット VTSIMNX_API_URL、VTSIMNX_API_KEY を使います。気象・風圧係数のファイルが一致した場合はノート内の検証済み結果を再利用します。指定フォルダに条件ごとの年間集計結果を保存し、再実行すると未完了の条件から再開します。等級・倍率の設定で、実行する条件を絞れます。

In [ ]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

weather_sha, wind_sha = digest(WEATHER_FILE), digest(WIND_COEFFICIENT_FILE)
reference_matches = (weather_sha, wind_sha) == (REFERENCE_WEATHER_SHA, REFERENCE_WIND_SHA)
run_id = hashlib.sha256(f"4e957e9-matrix-v1:{weather_sha}:{wind_sha}".encode()).hexdigest()[:16]
work = RESULT_DIR / run_id
if RUN_ANNUAL:
    work.mkdir(parents=True, exist_ok=True)

def save_checkpoint(path, payload):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False))
    tmp.replace(path)

seeded = {(row["等級"], row["倍率"]): row for row in VERIFIED_ANNUAL} if reference_matches else {}
rows = []
for grade in GRADES:
    for multiplier, c_value in h.GAP_CASES.items():
        case = {"等級": grade, "倍率": multiplier, "公称C値 [cm²/m²]": c_value}
        checkpoint = work / f"g{grade}_x{multiplier:g}.json"
        if checkpoint.exists():
            saved = json.loads(checkpoint.read_text())
            if saved["等級"] != grade or saved["倍率"] != multiplier or saved["状態"] != "完了":
                raise ValueError(f"条件と保存済み結果が一致しません {checkpoint}")
            row = {**case, **saved}
        elif (grade, multiplier) in seeded:
            row = {**case, **seeded[grade, multiplier]}
            if RUN_ANNUAL:
                save_checkpoint(checkpoint, row)
        elif RUN_ANNUAL and grade in GRADES_TO_RUN and multiplier in MULTIPLIERS_TO_RUN:
            print(f"等級{grade}　倍率×{multiplier:g}　年間計算", flush=True)
            try:
                config = make_grade_input(grade, multiplier)
                result = vt.run_calc(
                    userdata.get("VTSIMNX_API_URL"), config,
                    api_key=userdata.get("VTSIMNX_API_KEY"), timeout=3600, raise_on_error=True,
                )
                loads, monthly = grade_loads(result, config)
                row = {**case, **loads, "状態": "完了"}
                monthly.to_csv(work / f"g{grade}_x{multiplier:g}_monthly.csv", encoding="utf-8-sig")
                save_checkpoint(checkpoint, row)
            except Exception as exc:
                row = {**case, "状態": f"{type(exc).__name__}: {exc}"}
            print(f"等級{grade}　倍率×{multiplier:g}　{row['状態']}", flush=True)
        else:
            row = {**case, "状態": "未計算"}
        rows.append(row)

annual = pd.DataFrame(rows).reindex(columns=["等級", "倍率", "公称C値 [cm²/m²]", *METRICS, "状態"])
for metric in (HEATING, COOLING, TOTAL, POWER):
    for column, value, key, label in (("倍率", 1, "等級", "同等級×1比"), ("等級", 4, "倍率", "同倍率等級4比")):
        reference = annual.loc[annual[column].eq(value), [key, metric]].set_index(key)[metric]
        base = annual[key].map(reference)
        annual[metric.replace(" [kWh]", f"・{label} [kWh]")] = annual[metric] - base
        annual[metric.replace(" [kWh]", f"・{label} [%]")] = (annual[metric] / base.where(base.ne(0)) - 1) * 100
if RUN_ANNUAL:
    annual.to_csv(work / "annual_24_cases.csv", index=False, encoding="utf-8-sig")
display(annual[["等級", "倍率", "公称C値 [cm²/m²]", *METRICS, "状態"]].round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, metric in zip(axes.flat, [HEATING, COOLING, TOTAL, POWER]):
    matrix = annual.pivot(index="倍率", columns="等級", values=metric).reindex(list(h.GAP_CASES))
    for grade in GRADES:
        ax.plot(range(len(matrix)), matrix[grade], marker="o", label=f"等級{grade}")
    ax.set(title=metric, ylabel="kWh/年", xticks=range(len(matrix)),
           xticklabels=[f"×{m:g}" for m in matrix.index], xlabel="外皮隙間倍率")
    ax.grid(alpha=0.3)
axes[0, 1].legend()
fig.tight_layout()
plt.show()

## 短期間の検証結果

年間計算と同じモデルの短期間計算です。冬は1月1〜7日（12月15日から助走）、夏は7月31日〜8月6日（7月24日から助走）の各168時間を集計しています。

In [ ]:
if reference_matches:
    weekly = pd.DataFrame(VERIFIED_WEEKS)
    display(weekly.sort_values(["期間", "等級", "倍率"]).round(2))